# Which credit-card customers will default next month?
30,000 customers of a Taiwanese bank, April to September 2005, with whether each defaulted in October 2005
(Yeh & Lien, 2009; UCI Machine Learning Repository, CC BY 4.0).

Run from the project folder:  `python src/get_data.py`  (once)  then  `python src/analysis.py`

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd()
FIG, RES = ROOT / "figures", ROOT / "results"
FIG.mkdir(exist_ok=True), RES.mkdir(exist_ok=True)
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8982"
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160, "savefig.bbox": "tight", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e4dd", "axes.axisbelow": True,
                     "font.size": 10.5, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left"})
funnel = []


def log(label, df, note):
    funnel.append({"label": label, "rows": int(len(df)), "note": note})
    print(f"{label:<45s} {len(df):>7,d} rows   {note}")

## 1. Load
The spreadsheet has two header rows. Row 0 holds codes (X1, X2 …); the real names are in row 1.
Reading with the default `header=0` gives a table where every column is text. `header=1` fixes it.

In [2]:
wrong = pd.read_excel(ROOT / "data" / "default_of_credit_card_clients.xls")
print("With header=0, numeric columns:", wrong.select_dtypes("number").shape[1], "of", wrong.shape[1])
df = pd.read_excel(ROOT / "data" / "default_of_credit_card_clients.xls", header=1)
df = df.rename(columns={"default payment next month": "default", "PAY_0": "PAY_1"})     # the file skips from PAY_0 to PAY_2
print("With header=1, numeric columns:", df.select_dtypes("number").shape[1], "of", df.shape[1])
log("Raw customers", df, "one row = one credit-card customer")

With header=0, numeric columns: 0 of 25


With header=1, numeric columns: 25 of 25
Raw customers                                  30,000 rows   one row = one credit-card customer


## 2. Clean
### 2.1 "No missing values" is not the same as "no problems"
`isna().sum()` is zero everywhere. The gaps are hiding as category codes the documentation never defines.

In [3]:
print("NaN cells:", int(df.isna().sum().sum()))
print("EDUCATION codes:", df["EDUCATION"].value_counts().sort_index().to_dict(), " documented: 1 graduate school, 2 university, 3 high school, 4 other")
print("MARRIAGE codes :", df["MARRIAGE"].value_counts().sort_index().to_dict(), " documented: 1 married, 2 single, 3 other")
undoc_edu, undoc_mar = int(df["EDUCATION"].isin([0, 5, 6]).sum()), int((df["MARRIAGE"] == 0).sum())
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})           # undocumented -> "other": 345 rows kept, not deleted
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})
print(f"Undocumented codes merged into 'other': EDUCATION {undoc_edu}, MARRIAGE {undoc_mar}")

NaN cells: 0
EDUCATION codes: {0: 14, 1: 10585, 2: 14030, 3: 4917, 4: 123, 5: 280, 6: 51}  documented: 1 graduate school, 2 university, 3 high school, 4 other
MARRIAGE codes : {0: 54, 1: 13659, 2: 15964, 3: 323}  documented: 1 married, 2 single, 3 other
Undocumented codes merged into 'other': EDUCATION 345, MARRIAGE 54


### 2.2 Repayment status has two undocumented values that most rows use
`PAY_n` is documented as -1 = paid in full, 1..9 = months late. But 0 and -2 appear in most rows.
The dataset's author later explained them: 0 = paid the minimum (revolving credit), -2 = no card use that month.
They are meaningful states, so they are kept. "Months late" is then a separate, clean number.

In [4]:
pay = [f"PAY_{i}" for i in range(1, 7)]
print("PAY_1 (September) values:", df["PAY_1"].value_counts().sort_index().to_dict())
print(f"Rows using the undocumented 0 or -2 in September: {df['PAY_1'].isin([0, -2]).mean():.1%}")
late = df[pay].clip(lower=0)
df["months_late_now"] = late["PAY_1"]
df["max_months_late"] = late.max(axis=1)
df["n_months_late"] = (late > 0).sum(axis=1)

PAY_1 (September) values: {-2: 2759, -1: 5686, 0: 14737, 1: 3688, 2: 2667, 3: 322, 4: 76, 5: 26, 6: 11, 7: 9, 8: 19}
Rows using the undocumented 0 or -2 in September: 58.3%


### 2.3 Thirty-five "duplicates" that are not duplicates
`duplicated()` flags 35 rows as copies of an earlier row (70 rows are involved in total). They have different
IDs, and 60 of the 70 are dormant accounts with zero bills in every month. Different customers can share a
profile; deleting them would be inventing a rule. They are kept.

In [5]:
dup = df.drop(columns="ID").duplicated(keep=False)
print("Flagged by duplicated():", int(df.drop(columns="ID").duplicated().sum()), "| rows involved:", int(dup.sum()), "| of which with all six bills = 0:", int((df.loc[dup, [f'BILL_AMT{i}' for i in range(1, 7)]].sum(axis=1) == 0).sum()))
print("Duplicate IDs:", int(df["ID"].duplicated().sum()))
log("After cleaning", df, f"{undoc_edu + undoc_mar} undocumented category codes recoded; 0 rows deleted")

Flagged by duplicated(): 35 | rows involved: 70 | of which with all six bills = 0: 60
Duplicate IDs: 0
After cleaning                                 30,000 rows   399 undocumented category codes recoded; 0 rows deleted


### 2.4 Features a credit analyst would ask for

In [6]:
bills, pays = [f"BILL_AMT{i}" for i in range(1, 7)], [f"PAY_AMT{i}" for i in range(1, 7)]
print("Negative bills (customer in credit):", int((df["BILL_AMT1"] < 0).sum()), "| bills above the limit:", int((df["BILL_AMT1"] > df["LIMIT_BAL"]).sum()))
df["utilisation"] = (df["BILL_AMT1"] / df["LIMIT_BAL"]).clip(-0.5, 2)              # share of the limit in use
df["avg_utilisation"] = (df[bills].mean(axis=1) / df["LIMIT_BAL"]).clip(-0.5, 2)
df["pay_ratio"] = (df["PAY_AMT1"] / df["BILL_AMT2"].where(df["BILL_AMT2"] > 0)).clip(0, 2)   # paid in Sept / billed in Aug; NaN when nothing was owed
df["pay_ratio_missing"] = df["pay_ratio"].isna().astype(int)
df["pay_ratio"] = df["pay_ratio"].fillna(1.0)                                      # nothing owed = fully paid; flagged above
df["log_limit"] = np.log10(df["LIMIT_BAL"])
df["bill_trend"] = ((df["BILL_AMT1"] - df["BILL_AMT6"]) / df["LIMIT_BAL"]).clip(-2, 2)
print(f"Default rate: {df['default'].mean():.2%}  ({df['default'].sum():,} of {len(df):,})")

Negative bills (customer in credit): 590 | bills above the limit: 2115
Default rate: 22.12%  (6,636 of 30,000)


## 3. Explore

In [7]:
st_lab = {-2: "no use", -1: "paid in full", 0: "paid minimum", 1: "1 month late", 2: "2 months late", 3: "3+ months late"}
by_pay = df.groupby(df["PAY_1"].clip(upper=3))["default"].agg(["mean", "size"])
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.bar([st_lab[k] for k in by_pay.index], by_pay["mean"] * 100, color=BLUE, width=0.7)
for x, (m, n) in enumerate(zip(by_pay["mean"], by_pay["size"])):
    ax.text(x, m * 100 + 1.5, f"{m:.0%}\nn={n:,}", ha="center", fontsize=8.5, color="#52514e")
ax.axhline(df["default"].mean() * 100, color=GREY, ls="--", lw=1)
ax.set(ylabel="% who defaulted in October", xlabel="Repayment status in September", ylim=(0, 92), title="Being two months late in September is the line that matters")
plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
fig.savefig(FIG / "fig1_default_by_status.png"); plt.close(fig)

In [8]:
lim_bins = [0, 30000, 50000, 100000, 200000, 300000, 1e7]
lim_lab = ["≤30k", "30-50k", "50-100k", "100-200k", "200-300k", ">300k"]
by_lim = df.groupby(pd.cut(df["LIMIT_BAL"], lim_bins, labels=lim_lab), observed=True)["default"].agg(["mean", "size"])
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.bar(lim_lab, by_lim["mean"] * 100, color=BLUE, width=0.7)
for x, (m, n) in enumerate(zip(by_lim["mean"], by_lim["size"])):
    ax.text(x, m * 100 + 0.8, f"{m:.0%}\nn={n:,}", ha="center", fontsize=8.5, color="#52514e")
ax.set(xlabel="Credit limit (New Taiwan dollars)", ylabel="% who defaulted", ylim=(0, by_lim["mean"].max() * 125), title="Customers given small limits default most: the bank already knew something")
fig.savefig(FIG / "fig2_default_by_limit.png"); plt.close(fig)

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.2))
axes[0].hist(df["BILL_AMT1"] / 1000, bins=60, color=BLUE); axes[0].set(xlabel="September bill (thousand NT$)", ylabel="customers", title=f"Raw (skew {df['BILL_AMT1'].skew():.1f})")
axes[1].hist(df["utilisation"], bins=60, color=BLUE); axes[1].set(xlabel="Bill ÷ credit limit", title=f"As utilisation (skew {df['utilisation'].skew():.1f})")
fig.savefig(FIG / "fig3_skew.png"); plt.close(fig)
by_demo = {c: df.groupby(c)["default"].mean().round(4).to_dict() for c in ["SEX", "EDUCATION", "MARRIAGE"]}
print(by_demo)

{'SEX': {1: 0.2417, 2: 0.2078}, 'EDUCATION': {1: 0.1923, 2: 0.2373, 3: 0.2516, 4: 0.0705}, 'MARRIAGE': {1: 0.2347, 2: 0.2093, 3: 0.2361}}


## 4. Split
There is one snapshot in time, so a time-based split is impossible (a limitation: see the paper).
A stratified random split keeps the 22% default rate identical in train and test.

In [10]:
behaviour = pay + ["months_late_now", "max_months_late", "n_months_late", "utilisation", "avg_utilisation", "pay_ratio", "pay_ratio_missing", "log_limit", "bill_trend"] + bills + pays
demographic = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["default"])
y_train, y_test = train["default"], test["default"]
print(f"train {len(train):,} ({y_train.mean():.2%} default) | test {len(test):,} ({y_test.mean():.2%} default)")

train 24,000 (22.12% default) | test 6,000 (22.12% default)


## 5. Models
Compared by 5-fold cross-validation on the training set; the test set is touched once at the end.
The main model uses **behaviour only**. Sex, marital status and age are legally protected characteristics in
lending in many countries, so the model is built without them and the cost of leaving them out is measured.

In [11]:
def logit(cols):
    cat = [c for c in cols if c in ("EDUCATION", "MARRIAGE", "SEX")]
    num = [c for c in cols if c not in cat]
    return make_pipeline(ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)]), LogisticRegression(max_iter=2000))


def gbm(cols):
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=4, l2_regularization=1.0, early_stopping=True, validation_fraction=0.15, random_state=42)


def score(name, p, y):
    return {"name": name, "auc": roc_auc_score(y, p), "ap": average_precision_score(y, p), "brier": brier_score_loss(y, p), "ks": float(ks_2samp(p[y == 1], p[y == 0]).statistic)}


cv = StratifiedKFold(5, shuffle=True, random_state=42)
results = [{"name": "Baseline: always the average rate", "auc": 0.5, "ap": float(y_test.mean()), "brier": brier_score_loss(y_test, np.full(len(test), y_train.mean())), "ks": 0.0}]
rule_p = (test["PAY_1"] >= 2).astype(float) * 0.6 + 0.1
results.append(score("One rule: 2+ months late in September", rule_p.values, y_test.values))
fitted = {}
for name, make, cols in [("Logistic regression, behaviour", logit, behaviour), ("Gradient boosting, behaviour", gbm, behaviour), ("Gradient boosting, behaviour + demographics", gbm, behaviour + demographic)]:
    cv_auc = cross_val_score(make(cols), train[cols], y_train, cv=cv, scoring="roc_auc")
    m = make(cols).fit(train[cols], y_train)
    p = m.predict_proba(test[cols])[:, 1]
    results.append({**score(name, p, y_test.values), "cv_auc_mean": float(cv_auc.mean()), "cv_auc_sd": float(cv_auc.std())}); fitted[name] = (m, cols, p)
    print(results[-1])

r = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.barh(r["name"][::-1], r["auc"][::-1], color=[BLUE, BLUE, BLUE, BLUE, GREY], height=0.65)
for y, v in enumerate(r["auc"][::-1]):
    ax.text(v + 0.005, y, f"{v:.3f}", va="center", fontsize=9)
ax.set(xlim=(0.45, 0.85), xlabel="ROC AUC on the 6,000 held-out customers", title="One rule gets part of the way; a model adds ranking; demographics add almost nothing")
ax.grid(axis="y", visible=False)
fig.savefig(FIG / "fig4_models.png"); plt.close(fig)

{'name': 'Logistic regression, behaviour', 'auc': 0.758648546355944, 'ap': 0.5217757184673312, 'brier': 0.13971339973757327, 'ks': 0.39307597026384633, 'cv_auc_mean': 0.7677532505889667, 'cv_auc_sd': 0.0032336618420295772}


{'name': 'Gradient boosting, behaviour', 'auc': 0.7786554935429701, 'ap': 0.5558241655162149, 'brier': 0.13482964851218407, 'ks': 0.42328478419292415, 'cv_auc_mean': 0.7837219651129497, 'cv_auc_sd': 0.004726038790526945}


{'name': 'Gradient boosting, behaviour + demographics', 'auc': 0.7810103286996714, 'ap': 0.5582683030147328, 'brier': 0.13463413174918068, 'ks': 0.43188926558009094, 'cv_auc_mean': 0.7853386640073519, 'cv_auc_sd': 0.004236392536387616}


In [12]:
best_name = "Gradient boosting, behaviour"
best_m, best_cols, best_p = fitted[best_name]
cal = pd.DataFrame({"p": best_p, "y": y_test.values}); cal["bin"] = pd.qcut(cal["p"], 10, duplicates="drop")
calib = cal.groupby("bin", observed=True).agg(pred=("p", "mean"), actual=("y", "mean"), n=("y", "size"))
print(calib.round(3))
fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.plot([0, 1], [0, 1], color=GREY, ls="--", lw=1); ax.plot(calib["pred"], calib["actual"], marker="o", color=ORANGE, lw=2)
ax.set(xlabel="Predicted probability of default", ylabel="Share who defaulted", title="Calibration (ten equal-sized groups)")
fig.savefig(FIG / "fig5_calibration.png"); plt.close(fig)
pi = permutation_importance(best_m, test[best_cols], y_test, scoring="roc_auc", n_repeats=8, random_state=1, n_jobs=2)
imp = pd.Series(pi.importances_mean, index=best_cols).sort_values(ascending=False)
nice = {"PAY_1": "Repayment status, September", "months_late_now": "Months late, September", "max_months_late": "Worst lateness in 6 months", "n_months_late": "Number of late months", "utilisation": "Share of limit in use",
        "avg_utilisation": "Average share of limit in use", "pay_ratio": "Paid ÷ billed last month", "log_limit": "Credit limit", "bill_trend": "Change in balance over 6 months", "pay_ratio_missing": "Nothing owed last month"}
nice.update({f"PAY_{i}": f"Repayment status, {m}" for i, m in zip(range(2, 7), ["August", "July", "June", "May", "April"])})
nice.update({f"BILL_AMT{i}": f"Bill, {m}" for i, m in zip(range(1, 7), ["September", "August", "July", "June", "May", "April"])})
nice.update({f"PAY_AMT{i}": f"Amount paid, {m}" for i, m in zip(range(1, 7), ["September", "August", "July", "June", "May", "April"])})
print(imp.head(10).round(4))

                   pred  actual    n
bin                                 
(0.0166, 0.0522]  0.041   0.042  600
(0.0522, 0.0758]  0.064   0.073  600
(0.0758, 0.102]   0.089   0.082  600
(0.102, 0.126]    0.114   0.115  600
(0.126, 0.148]    0.137   0.175  600
(0.148, 0.178]    0.162   0.153  600
(0.178, 0.226]    0.199   0.193  600
(0.226, 0.337]    0.272   0.249  603
(0.337, 0.586]    0.430   0.432  597
(0.586, 0.883]    0.705   0.698  600


PAY_1              0.0332
max_months_late    0.0290
utilisation        0.0081
BILL_AMT1          0.0069
avg_utilisation    0.0050
log_limit          0.0047
n_months_late      0.0035
PAY_AMT1           0.0034
PAY_AMT2           0.0034
PAY_AMT3           0.0033
dtype: float64


## 6. The decision: whom does the collections team call?
A missed defaulter costs the bank far more than an unnecessary call costs. With a cost ratio the threshold
stops being 0.5 by habit and becomes the value that minimises expected cost.

In [13]:
order = np.argsort(-best_p); yt = y_test.values[order]
decile = [{"flag_top": s, "customers": int(len(yt) * s), "precision": float(yt[: int(len(yt) * s)].mean()), "recall": float(yt[: int(len(yt) * s)].sum() / yt.sum())} for s in (0.1, 0.2, 0.3, 0.5)]
print(pd.DataFrame(decile).round(3))
COST_FN, COST_FP = 5.0, 1.0                          # illustrative: a missed default costs five times an unnecessary intervention
thr = np.linspace(0.02, 0.9, 89)
cost = [COST_FN * ((best_p < t) & (y_test.values == 1)).sum() + COST_FP * ((best_p >= t) & (y_test.values == 0)).sum() for t in thr]
t_best = float(thr[int(np.argmin(cost))])
at = lambda t: {"threshold": float(t), "flagged": int((best_p >= t).sum()), "caught": int(((best_p >= t) & (y_test.values == 1)).sum()), "missed": int(((best_p < t) & (y_test.values == 1)).sum()),
                "false_alarms": int(((best_p >= t) & (y_test.values == 0)).sum()), "cost": float(COST_FN * ((best_p < t) & (y_test.values == 1)).sum() + COST_FP * ((best_p >= t) & (y_test.values == 0)).sum())}
print("habit 0.5:", at(0.5)); print("cost-minimising:", at(t_best)); print("do nothing cost:", COST_FN * y_test.sum(), "| call everyone cost:", COST_FP * (1 - y_test).sum())
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(thr, cost, color=BLUE, lw=2.2); ax.axvline(t_best, color=ORANGE, lw=1.5); ax.axvline(0.5, color=GREY, ls="--", lw=1)
ax.text(t_best + 0.01, max(cost) * 0.97, f"lowest cost\nat {t_best:.2f}", color=ORANGE, va="top", fontsize=9); ax.text(0.51, max(cost) * 0.97, "default 0.5", color=GREY, va="top", fontsize=9)
ax.set(xlabel="Probability threshold above which a customer is contacted", ylabel="Total cost (units of one call)", title="If a missed default costs 5 calls, the best threshold is far below 0.5")
fig.savefig(FIG / "fig6_threshold_cost.png"); plt.close(fig)

   flag_top  customers  precision  recall
0       0.1        600      0.698   0.316
1       0.2       1200      0.566   0.512
2       0.3       1800      0.459   0.623
3       0.5       3000      0.345   0.780
habit 0.5: {'threshold': 0.5, 'flagged': 728, 'caught': 484, 'missed': 843, 'false_alarms': 244, 'cost': 4459.0}
cost-minimising: {'threshold': 0.16999999999999998, 'flagged': 2536, 'caught': 971, 'missed': 356, 'false_alarms': 1565, 'cost': 3345.0}
do nothing cost: 6635.0 | call everyone cost: 4673.0


In [14]:
edges = np.linspace(0, 1, 51)
posh, _ = np.histogram(best_p[y_test.values == 1], bins=edges); negh, _ = np.histogram(best_p[y_test.values == 0], bins=edges)
out = {"n_raw": int(len(df)), "default_rate": float(df["default"].mean()), "undoc_edu": undoc_edu, "undoc_mar": undoc_mar, "pay_undoc_share": float(df["PAY_1"].isin([0, -2]).mean()),
       "dup_rows": int(dup.sum()), "neg_bills": int((df["BILL_AMT1"] < 0).sum()), "over_limit": int((df["BILL_AMT1"] > df["LIMIT_BAL"]).sum()), "skew_bill": float(df["BILL_AMT1"].skew()), "skew_util": float(df["utilisation"].skew()),
       "train": {"n": int(len(train)), "rate": float(y_train.mean())}, "test": {"n": int(len(test)), "rate": float(y_test.mean())}, "models": results, "funnel": funnel,
       "by_pay": [{"bin": st_lab[k], "rate": float(v["mean"]), "n": int(v["size"])} for k, v in by_pay.iterrows()], "by_limit": [{"bin": str(k), "rate": float(v["mean"]), "n": int(v["size"])} for k, v in by_lim.iterrows()],
       "by_demo": {k: {str(a): float(b) for a, b in v.items()} for k, v in by_demo.items()},
       "calibration": [{"pred": float(x.pred), "actual": float(x.actual), "n": int(x.n)} for x in calib.itertuples()], "importance": [{"name": nice.get(k, k), "value": float(v)} for k, v in imp.items()],
       "decile": decile, "cost": {"fn": COST_FN, "fp": COST_FP, "habit": at(0.5), "best": at(t_best), "do_nothing": float(COST_FN * y_test.sum()), "call_everyone": float(COST_FP * (1 - y_test).sum())},
       "bins": {"pos": posh.tolist(), "neg": negh.tolist()}}
(RES / "metrics.json").write_text(json.dumps(out, indent=1))
print("Saved figures/ and results/metrics.json")

Saved figures/ and results/metrics.json
